# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 13 — Unsupervised Learning: Exercise Solutions</div>
<div align="center"> Jonathan Holmes (he/him)</div>

These exercises use the simulated four-blob dataset from Part 1 (`make_blobs` with 150 observations and 4 true centers) and the `USArrests` dataset for hierarchical clustering and PCA.

# In-Class Exercise #1 — K-Means Clustering

**Q1. In the K = 2 solution above, the algorithm splits the data into two groups even though the true data has four clusters. Why does K-means produce a result like this when $K$ is misspecified? What would you expect to happen to total within-cluster variation as $K$ increases toward 4?**

---

**Answer.**

K-means is told *exactly* how many clusters to find — that is the entire role of the $K$ argument. It does not look at the data and decide for itself that there should be four groups; it minimizes the within-cluster sum of squares (WCSS) **subject to the constraint that there are $K$ clusters**. So when $K = 2$ on data that really has four blobs, the algorithm picks the partition into two groups that gives the lowest WCSS — typically by merging the two nearest pairs of true blobs into two larger super-clusters. The result looks geometrically reasonable but is not the underlying structure.

**Total within-cluster variation as $K$ rises:**

$$\text{WCSS}(K) \;=\; \sum_{k=1}^K \sum_{i \in C_k} \|x_i - \bar x_k\|^2.$$

WCSS is monotonically **non-increasing** in $K$ — adding another cluster gives the algorithm strictly more flexibility (it can always keep the previous solution as a special case), so it can only reduce or keep the same total within-cluster distance. Concretely:

* $K = 1$: WCSS equals the total variance of the data.
* $K = 2$: WCSS drops sharply because the algorithm separates the two largest groups of points.
* $K = 3$: WCSS drops further — splitting one of the merged super-clusters reveals real structure.
* $K = 4$: WCSS drops again, hitting a clear *elbow*: each true blob is its own cluster, so further splits would cut into noise rather than into real groups.
* $K \geq 5$: WCSS keeps decreasing slowly as the algorithm subdivides individual blobs, but the marginal gain per extra cluster is small.

The plot of WCSS against $K$ is the **elbow plot**, and the bend at $K = 4$ is the visual signal that four is the natural number of clusters here. Note that WCSS *never* picks $K$ automatically — it only ever decreases — so we have to inspect the elbow ourselves.

**Q2. The K-means algorithm starts from a random initialisation. Why might two runs with the same $K$ return different cluster assignments? How does the `n_init` parameter address this?**

---

**Answer.**

K-means is a *greedy, iterative* algorithm: at each step it makes the locally-best move (re-assign points to the nearest center, then move each center to its cluster mean), and it stops when assignments stop changing. Greedy iteration is fast, but it does not guarantee a global optimum — it converges to a *local* minimum of the WCSS objective. **Which local minimum it lands in depends on where it started**, i.e., the random initial choice of cluster centers.

On well-separated data (like our four blobs) most random starts will find the same answer. But when clusters are close together, partially overlapping, or unevenly sized, different initializations can produce noticeably different final solutions — sometimes splitting a real cluster in two, sometimes merging two real clusters, all while giving plausible-looking outputs.

**`n_init` is the practical fix.** Setting `n_init=20` (for example) tells scikit-learn to run K-means **20 independent times** from 20 different random initializations and return the solution with the lowest WCSS. Because the WCSS objective is the same across runs, we can compare them on a single number and pick the best. With enough random starts, the chance that *all* of them get stuck in the same poor local minimum becomes vanishingly small.

Defaults in scikit-learn used to be `n_init=10`; in modern versions it has moved to `n_init="auto"`. Either way, the takeaway is: **never rely on a single K-means run**, and report the WCSS of the chosen solution so a reader can see how good the local minimum was.

**Q3. Modify the code to try $K = 5$ and $K = 6$. Does the solution look sensible? What does this suggest about the challenge of choosing $K$?**

---

**Answer.**

With $K = 5$ or $K = 6$ on data that really has four blobs, K-means *will* find five or six clusters — it has no choice — but the extra clusters are not finding real new groups. Typical patterns to expect:

* The algorithm **splits one of the true blobs in two**, drawing a somewhat arbitrary line through it. The exact line depends on the random initialization, so the split looks different on different runs.
* The split tends to fall through the *largest* or *most diffuse* blob, because that is where there is the most within-cluster variance to chip away at.
* WCSS keeps dropping a little further than at $K = 4$, but the marginal gain per added cluster is much smaller than the gains from $K = 1 \to 2 \to 3 \to 4$.

**What this teaches us about choosing $K$:**

1. **K-means cannot tell you the right $K$.** Its objective only ever favours more clusters, so any rule that just minimizes WCSS will pick $K = n$ (one cluster per observation, WCSS = 0).
2. **Choosing $K$ requires a separate criterion.** Common ones are the **elbow rule** (look at the WCSS-vs-$K$ plot for a bend), the **silhouette score** (a measure of how well each point fits its cluster vs. neighbouring clusters), the **gap statistic**, or domain knowledge (e.g., "we expect three customer segments").
3. **Several values of $K$ may be defensible.** On real data the elbow is often gentle rather than sharp, and reasonable analysts can pick $K = 3$ or $K = 5$ for the same dataset. Reporting the choice and the criterion that produced it is the right standard.

Unsupervised learning is harder than supervised learning precisely because there is no held-out *correct answer* to validate against — only a model fit and your judgment about whether it tells a sensible story.

# In-Class Exercise #2 — Hierarchical Clustering

**Q4. Set `n_clusters = 4` in the `fcluster` cell and compare the resulting scatter plot to the K-means K=4 result from Part 1. Do the two methods agree on where the cluster boundaries lie? Where do they disagree?**

---

**Answer.**

Roughly: the two methods **agree on the broad partition** but **disagree on a handful of borderline observations**. Both correctly recover the four true blobs as four distinct groups, with assignments matching for the vast majority of points.

Where they tend to disagree:

* **At the boundaries between adjacent blobs.** A point that lies roughly halfway between two centers is a *coin flip* — K-means assigns it to whichever centroid is marginally closer (Euclidean distance to a single centroid), while complete-linkage hierarchical clustering compares it to the *farthest* point in each candidate cluster. These two tiebreakers can disagree.
* **For outliers.** An observation that is far from any blob may end up in its own dendrogram branch under hierarchical clustering, but get pulled into the nearest blob under K-means.

Why the disagreement at the boundary, mechanistically: K-means optimizes a global objective (within-cluster sum of squares), so cluster boundaries fall along **perpendicular bisectors** between centroids — straight lines in 2D. Hierarchical clustering with complete linkage builds the partition *bottom up* from pairwise distances, so cluster boundaries follow the *merge sequence*, not a single global criterion, and can have more irregular shapes.

On well-separated synthetic data the disagreements are minor. On real data with overlapping clusters and noise, the two methods can produce visibly different partitions — which is why it is good practice to try both and check that the qualitative story is robust to the choice of method.

**Q5. Now try `n_clusters = 2` and `n_clusters = 3`. Looking at the dendrogram, at what heights do the cuts fall? Do the cluster boundaries make intuitive sense in the scatter plot?**

---

**Answer.**

The `fcluster(..., criterion='maxclust')` call cuts the dendrogram at the height that produces *exactly* the requested number of clusters. Mechanically, that height sits between the merge that **creates** the desired count and the next merge **above** it (which would reduce the count by one).

For our four-blob data:

* **`n_clusters = 4`** cuts low in the dendrogram, at the height of the merge that joined the two closest blobs — somewhere just below the four big top-level branches.
* **`n_clusters = 3`** cuts slightly higher: one of the four branches has already been merged with its neighbour at that height, so we get three super-clusters, two of which are pure blobs and one of which is a fused pair.
* **`n_clusters = 2`** cuts very high — near the top of the dendrogram, just below the final merge. The four blobs collapse into two super-clusters, each containing two of the original blobs.

**Do the boundaries make sense?** Yes, in the geometric sense: the two super-clusters at $K = 2$ correspond to the two halves of the scatter plot that are farthest apart, and the three-cluster solution merges the two closest blobs first. Whether this is *substantively* sensible depends on what the blobs represent — geometrically, the algorithm is doing the right thing, but only the analyst can decide whether four blobs really should be lumped into two for the purposes of a particular question.

Useful diagnostic: look for **vertical gaps** between merges in the dendrogram. A long vertical line between two merges means the merge is joining two genuinely distant clusters; cutting just below that long line picks out a natural number of groups. The four-blob data should show a clear gap between the merges that join blobs *within* a super-cluster and the much taller merges that join one super-cluster to another.

**Q6. Change `method='complete'` to `method='single'` in the `linkage` call and replot the dendrogram. How does the shape change, and why? (Hint: think about what single linkage does when one observation is close to a large cluster.)**

---

**Answer.**

**Single linkage** defines the distance between two clusters as the *minimum* pairwise distance between any observation in one and any observation in the other:

$$d_{\text{single}}(A, B) = \min_{i \in A,\; j \in B} \|x_i - x_j\|.$$

**Complete linkage** uses the *maximum* pairwise distance instead. The two definitions usually behave very differently:

* **Complete linkage** produces *compact, balanced* clusters because it requires every pair of points across the merged cluster to be reasonably close. The dendrogram tends to look symmetric, with merges happening at intermediate heights.
* **Single linkage** produces *long, stringy, chained* clusters. As soon as one observation in cluster $A$ is close to *one* observation in cluster $B$, the two clusters merge — even if every other pair across $A$ and $B$ is far apart. The dendrogram develops a characteristic **staircase** shape: many low merges as one big cluster grows by absorbing nearby observations one at a time.

On our four-blob data, single linkage often produces a dendrogram where one big chain dominates, with most observations attached to it through low-height merges, and only a few late merges at higher heights. The cluster structure becomes harder to read, and a horizontal cut at a sensible height may produce one giant cluster plus several singletons — not the four clean groups we want.

Single linkage's chaining behaviour makes it sensitive to noise: a single bridge-point between two clusters can collapse them into one. That is why **complete and average linkage are the standard defaults** in practice; single linkage is mostly used when the goal really is to detect chains (e.g., in connected-components problems) rather than compact groups.

# In-Class Exercise #3 — Principal Component Analysis

**Q7. Look at the `pca_loadings` table. Murder, Assault, and Rape all load heavily onto the first principal component, while UrbanPop loads more onto the second. What does each principal component appear to be measuring in terms of the original variables?**

---

**Answer.**

Loadings tell us **how much each original variable contributes** to a principal component. Reading them, we can give each PC an interpretation:

* **PC1 — "Overall violent crime."** Murder, Assault, and Rape all have large positive loadings on PC1, while UrbanPop's loading is small. PC1 is essentially a *single composite index of violent crime*: a weighted average of the three crime variables that moves up when all of them move up. A state's PC1 score is high when its violent-crime rates are high, regardless of how urban it is.
* **PC2 — "Urbanisation."** The pattern reverses on PC2: UrbanPop has the largest loading, while the three crime variables have small (often negative) loadings. PC2 is essentially an *urban–rural axis* that is approximately uncorrelated with the crime axis after we have removed PC1's variation.

PCA has done two useful things for us:

1. **Reduced dimensionality.** The four original variables can be summarized by two principal components — *crime* and *urbanisation* — that together explain most of the variation in the dataset.
2. **Decorrelated the axes.** PC1 and PC2 are orthogonal by construction, so we can talk about a state's crime profile separately from its urbanisation profile, even though the original four variables are correlated with each other.

**Important caveats.** The names "violent crime" and "urbanisation" are *interpretive labels* we attach to the PCs based on which variables load on them — they are not built into the algorithm. PCA only tells us which linear combinations of the variables capture the most variance; whether those combinations correspond to a meaningful real-world concept is a judgment call. Always sanity-check your interpretation by examining a few states whose scores you have intuitions about (Q8 does exactly this).

**Q8. In the biplot, states like Florida, Nevada, and California appear far to the right (high PC1 scores). What does a high PC1 score imply about a state's crime profile?**

---

**Answer.**

From Q7, PC1 is essentially a violent-crime index. So a state far to the right of the biplot — high PC1 score — has **above-average rates of murder, assault, and rape simultaneously**, relative to the other 49 states. The interpretation is *cumulative*: high PC1 does not mean exceptionally high on one crime and average on the others; it means the state shows up as elevated on the *whole bundle* of violent-crime measures that load on PC1.

Reading the example states:

* **Florida, Nevada, California** at the far right are the states the algorithm identifies as having the highest overall violent-crime profile (within the 1973 USArrests data the dataset records). Their PC1 scores are several units above the mean.
* States at the far **left** (e.g., North Dakota, Vermont, New Hampshire) have *below-average* violent-crime rates across all three crime variables. Their PC1 scores are negative.
* States near the middle of the PC1 axis are roughly average on overall violent crime.

Two things to keep in mind:

1. **PC1 score is a relative ranking, not an absolute level.** It is computed from standardized variables, so "high PC1" means "high crime *relative to the average state in this sample*." A state with PC1 = 0 is at the average, not at zero crime.
2. **The sign of a principal component is arbitrary** — multiplying all PC1 loadings and scores by $-1$ gives a mathematically equivalent solution. Some PCA implementations may produce loadings of the opposite sign; in that case high PC1 would mean *low* crime. Always check the loading signs before interpreting the scores.

PC2 in the biplot tells the complementary story: states near the top (high PC2) tend to be more urbanized, regardless of where they sit on the crime axis. So Florida and California sit far right *and* fairly high — high crime *and* highly urban — while North Dakota sits far left and low — low crime, less urban. The two-component biplot lets us read a state's crime/urbanisation profile in a single picture.

**Q9. The scree plot shows that PC1 alone explains a large fraction of the variance. If you were building a regression model to predict something about U.S. states, at what point would you stop adding principal components? What criterion would you use?**

---

**Answer.**

There is no single right answer; several rules of thumb exist, and which one to use depends on what you are doing with the components.

**Common criteria (in roughly increasing order of formality):**

1. **Elbow rule (scree plot).** Plot proportion of variance explained against component number and look for a *bend* — the point where the curve flattens out. Components before the elbow capture meaningful structure; components after the elbow are mostly noise. On `USArrests`, PC1 explains about 62%, PC2 about 25%, and PC3 and PC4 are much smaller — the elbow falls between PC2 and PC3, suggesting we keep two components.
2. **Cumulative variance threshold.** Keep enough components to explain a target fraction of total variance — common choices are 80%, 90%, or 95%. On `USArrests`, two components explain $\approx 87\%$ of the variance; three components push us over 95%. Two is the natural choice.
3. **Kaiser criterion.** Keep components with *eigenvalue greater than 1* (equivalently, variance greater than the variance of a single standardized variable). Components below this threshold explain less variance than a single original variable, so they arguably are not earning their keep.
4. **Cross-validated predictive performance.** If the components are inputs to a downstream supervised model, use CV to choose the number that minimizes test error. This is the most rigorous method when prediction is the goal — it sidesteps the question of "how much variance in $X$" entirely and asks "how much do these PCs help me predict $Y$."

**Important warning specific to using PCA before a regression.** PCA is *unsupervised* — it ranks components by variance in $X$, not by their relevance to $Y$. It is entirely possible (and on real data, common) for the most predictive direction to live in PC3, PC4, or even later components. So the elbow / variance-threshold rules tell you which components capture variation in *the predictors*, but they do not guarantee that those components are the ones that matter for $Y$.

**Practical recommendation.** If you are using PCA primarily to *describe* and *visualize* the predictors, use the elbow or 80% rule and pick a small number of components for plotting and interpretation. If you are using PCA to *reduce dimensionality before a regression*, use cross-validation on the downstream model to choose the number of components — and consider whether a *supervised* dimensionality-reduction method (Partial Least Squares, or just regularized regression on the original variables) would be better suited to the task.